# Milestone 4: Baselines, Feature Ablations, and Model Evaluation

This notebook evaluates pre-release features for forecasting Asian drama success (`target_pct` percentile within country-year cohort, and 3-class `target_tier`: Low < 25%, Medium 25-75%, High >= 75%).

### Evaluation Protocol & Methodological Guardrails
1. **Strict Temporal Split:** Train = dramas released $\le 2021$ ($N = 5,010$); Test = dramas released $\ge 2022$ ($N = 2,525$).
2. **Internal Validation Inside Train:** Model selection, hyperparameter tuning, and permutation importance are conducted strictly inside train ($\le 2019$ train with $N=3,520$ vs $2020–2021$ val with $N=1,490$). The test set is never touched during tuning.
3. **Primary Matrix:** Variant 2 (V2, static train-only priors) was declared as the headline benchmark before test evaluation and is kept as headline even though "all groups" scores slightly higher.
4. **Controlled Subsets Evaluated:**
   - **Full Test** ($N = 2,525$, High base rate: 25.3%)
   - **complete_info** ($N = 1,022$, High base rate: 38.9%): No missing core attributes.
   - **complete_info_strict** ($N = 635$, High base rate: 44.1%): `complete_info` AND content rating known AND aired_on known AND genres known.
   - **Per-Country Slices:** South Korea ($N=751$), Thailand ($N=790$), China ($N=532$), Japan ($N=452$).
5. **Statistical Rigor & Limitations:**
   - 500 bootstrap resamples on test metrics; 500 paired bootstrap resamples for model-vs-model comparisons.
   - *Note on Bootstrap Independence:* Bootstrap resamples treat dramas as independent observations although dramas share directors, cast, and sequels. Consequently, confidence intervals are likely somewhat narrow / optimistic.
   - *Multiple Testing Note:* Many comparisons were run, so results with an interval that barely excludes zero should be treated as suggestive rather than conclusive.
   - *Metric Reliability Note:* On `complete_info` and strict, macro-F1 is unstable because the Low class represents only ~7.5% to 9.5% of the data; ROC-AUC, PR-AUC, and Spearman correlation are the primary metrics there.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from src.modeling import (
    load_modeling_data,
    get_temporal_validation_split,
    get_expanding_window_folds,
    compute_metrics,
    bootstrap_evaluation,
    paired_bootstrap_comparison,
    MajorityClassBaseline,
    StratifiedRandomBaseline,
    CountryOnlyBaseline,
    PageCompletenessBaseline,
    HGB_TUNED_PARAMS,
    train_and_predict_hgb,
    train_and_predict_linear,
)
from src.features import get_feature_columns, FEATURE_GROUPS

print("Python executable:", sys.executable)
print(f"Scikit-learn version: {sklearn.__version__}")

# Load V2 Headline and V1 Deployment benchmarks
train_df, test_df = load_modeling_data("v2")
train_v1, test_v1 = load_modeling_data("v1")

for df in [train_df, test_df, train_v1, test_v1]:
    df["complete_info_strict"] = (
        (df["complete_info"] == 1) &
        (df["content_rating_is_missing"] == 0) &
        (df["aired_on_is_missing"] == 0) &
        (df["genre_is_missing"] == 0)
    ).astype(int)

print(f"Train dramas (<=2021): {len(train_df):,} | Test dramas (>=2022): {len(test_df):,}")
print(f"Test complete_info:        {int(test_df['complete_info'].sum()):,} ({test_df['complete_info'].mean():.1%})")
print(f"Test complete_info_strict: {int(test_df['complete_info_strict'].sum()):,} ({test_df['complete_info_strict'].mean():.1%})")

Python executable: C:\Users\DELL\Downloads\Drama Success Copilot\venv\Scripts\python.exe
Scikit-learn version: 1.9.1


Train dramas (<=2021): 5,010 | Test dramas (>=2022): 2,525
Test complete_info:        1,022 (40.5%)
Test complete_info_strict: 635 (25.1%)


## Validation Protocol: Time-Ordered Folds Within Train

In [2]:
tr_sub, val_sub = get_temporal_validation_split(train_df, val_cutoff_year=2019)
print(f"Internal Train (<=2019): {len(tr_sub):,} dramas")
print(f"Internal Val (2020-2021): {len(val_sub):,} dramas")

folds = get_expanding_window_folds(train_df)
print("\nExpanding-Window Folds inside Train:")
for f in folds:
    print(f"  Fold {f['fold']}: Train {f['train_years']} (N={f['train_size']:,}) -> Val {f['val_years']} (N={f['val_size']:,})")

Internal Train (<=2019): 3,520 dramas
Internal Val (2020-2021): 1,490 dramas

Expanding-Window Folds inside Train:
  Fold 1: Train <= 2017 (N=2,401) -> Val 2018 (N=711)
  Fold 2: Train <= 2018 (N=3,112) -> Val 2019 (N=408)
  Fold 3: Train <= 2019 (N=3,520) -> Val 2020-2021 (N=1,490)


## Step 1: Foundational Baseline Models

In [3]:
y_train = train_df["target_tier"]

b_maj = MajorityClassBaseline().fit(train_df, y_train)
b_strat = StratifiedRandomBaseline(random_state=42).fit(train_df, y_train)
b_country = CountryOnlyBaseline(random_state=42).fit(train_df, y_train)
b_page = PageCompletenessBaseline(random_state=42).fit(train_df, y_train)

baselines = {
    "Majority Class (Medium)": b_maj,
    "Stratified Random": b_strat,
    "Country-Only": b_country,
    "Page Completeness Only": b_page,
}

def evaluate_baseline_dict(df_slice: pd.DataFrame) -> pd.DataFrame:
    y_true = df_slice["target_tier"].values
    t_pct = df_slice["target_pct"].values
    base_rate = (y_true == "High").mean()
    rows = []
    for name, model in baselines.items():
        preds = model.predict(df_slice)
        probs = model.predict_proba_high(df_slice)
        summary = bootstrap_evaluation(y_true, preds, probs, t_pct, n_boot=500, random_state=42)
        rows.append({
            "Baseline Model": name,
            "Accuracy": summary["Accuracy"]["str"],
            "Macro-F1": summary["Macro-F1"]["str"],
            "Balanced Acc": summary["Balanced Acc"]["str"],
            "ROC-AUC (High)": summary["ROC-AUC (High)"]["str"],
            f"PR-AUC [base:{base_rate:.3f}]": summary["PR-AUC (High)"]["str"],
            "Spearman": summary["Spearman"]["str"],
        })
    return pd.DataFrame(rows)

print("=== Step 1 Baselines: Full Test Set (N=2,525) ===")
display(evaluate_baseline_dict(test_df))

print("\n=== Step 1 Baselines: complete_info Subset (N=1,022) ===")
display(evaluate_baseline_dict(test_df[test_df["complete_info"] == 1].reset_index(drop=True)))

=== Step 1 Baselines: Full Test Set (N=2,525) ===

,Baseline Model,Accuracy,Macro-F1,Balanced Acc,ROC-AUC (High),PR-AUC [base:0.253],Spearman
0,Majority Class (Medium),"0.499 [0.479, 0.521]","0.222 [0.216, 0.228]","0.333 [0.333, 0.333]","0.500 [0.500, 0.500]","0.253 [0.237, 0.270]",N/A
1,Stratified Random,"0.365 [0.345, 0.381]","0.324 [0.304, 0.341]","0.324 [0.304, 0.341]","0.509 [0.483, 0.533]","0.257 [0.236, 0.280]","0.026 [-0.011, 0.065]"
2,Country-Only,"0.499 [0.479, 0.521]","0.222 [0.216, 0.228]","0.333 [0.333, 0.333]","0.501 [0.477, 0.526]","0.253 [0.234, 0.272]","0.002 [-0.038, 0.042]"
3,Page Completeness Only,"0.528 [0.509, 0.548]","0.502 [0.483, 0.524]","0.491 [0.472, 0.512]","0.736 [0.714, 0.754]","0.408 [0.375, 0.437]","0.516 [0.481, 0.548]"



=== Step 1 Baselines: complete_info Subset (N=1,022) ===


,Baseline Model,Accuracy,Macro-F1,Balanced Acc,ROC-AUC (High),PR-AUC [base:0.389],Spearman
0,Majority Class (Medium),"0.516 [0.483, 0.547]","0.227 [0.217, 0.236]","0.333 [0.333, 0.333]","0.500 [0.500, 0.500]","0.389 [0.359, 0.420]",N/A
1,Stratified Random,"0.364 [0.333, 0.395]","0.300 [0.270, 0.326]","0.313 [0.278, 0.346]","0.512 [0.474, 0.545]","0.401 [0.365, 0.449]","0.046 [-0.011, 0.105]"
2,Country-Only,"0.516 [0.483, 0.547]","0.227 [0.217, 0.236]","0.333 [0.333, 0.333]","0.408 [0.376, 0.441]","0.348 [0.317, 0.382]","-0.243 [-0.296, -0.186]"
3,Page Completeness Only,"0.489 [0.459, 0.520]","0.341 [0.320, 0.363]","0.374 [0.352, 0.395]","0.569 [0.538, 0.599]","0.427 [0.392, 0.465]","0.168 [0.106, 0.230]"


## Step 2: Feature Group Ablations & Predictive Modeling

In [4]:
feature_sets = {
    "1. core": get_feature_columns(["core"]),
    "2. core + talent_reach (Headline)": get_feature_columns(["core", "talent_reach"]),
    "3. core + talent_reach + missingness": get_feature_columns(["core", "talent_reach", "missingness"]) + ["n_missing_credits"],
    "4. core + talent_reach + talent_exp": get_feature_columns(["core", "talent_reach", "talent_experience"]),
    "5. core + talent_reach + text": get_feature_columns(["core", "talent_reach", "text"]),
    "6. talent_reach only": get_feature_columns(["talent_reach"]),
    "7. all groups": get_feature_columns(["core", "talent_reach", "missingness", "talent_experience", "text"]) + ["n_missing_credits"],
}

predictions = {}

# Page completeness baseline
pc_clf = PageCompletenessBaseline(random_state=42).fit(train_df, train_df["target_tier"])
predictions["Page Completeness Baseline"] = (pc_clf.predict(test_df), pc_clf.predict_proba_high(test_df))

# 1-7. GBDT feature group ablations
for name, cols in feature_sets.items():
    predictions[name] = train_and_predict_hgb(train_df, test_df, cols, params=HGB_TUNED_PARAMS)

# 8. Drop 2015 burn-in
tr_no2015 = train_df[train_df["release_year"] >= 2016].reset_index(drop=True)
predictions["8. core + talent_reach (Drop 2015)"] = train_and_predict_hgb(
    tr_no2015, test_df, feature_sets["2. core + talent_reach (Headline)"], params=HGB_TUNED_PARAMS
)

# 9. V1 vs V2 priors
predictions["9. core + talent_reach (V1 rolling)"] = train_and_predict_hgb(
    train_v1, test_v1, feature_sets["2. core + talent_reach (Headline)"], params=HGB_TUNED_PARAMS
)

# 10. Linear Reference (Logistic Regression on Headline)
predictions["10. Linear Reference (core + reach)"] = train_and_predict_linear(
    train_df, test_df, feature_sets["2. core + talent_reach (Headline)"]
)

print(f"Trained all {len(predictions)} model variants successfully!")

Trained all 11 model variants successfully!


In [5]:
def evaluate_models_slice(mask: np.ndarray, slice_title: str) -> pd.DataFrame:
    y_true = test_df.loc[mask, "target_tier"].values
    t_pct = test_df.loc[mask, "target_pct"].values
    base_rate = float((y_true == "High").mean())
    n = len(y_true)
    
    rng = np.random.RandomState(42)
    boot_indices = [rng.choice(n, size=n, replace=True) for _ in range(500)]
    
    rows = []
    for model_name, (pred, prob) in predictions.items():
        p_sub = pred[mask]
        pr_sub = prob[mask]
        pt = compute_metrics(y_true, p_sub, pr_sub, t_pct)
        
        boot_list = {k: [] for k in pt.keys()}
        for b_idx in boot_indices:
            b_m = compute_metrics(y_true[b_idx], p_sub[b_idx], pr_sub[b_idx], t_pct[b_idx])
            for k, v in b_m.items():
                boot_list[k].append(v)
                
        def get_ci_str(metric_name: str, fmt: str = ".3f") -> str:
            val = pt[metric_name]
            arr = np.array(boot_list[metric_name])
            arr_valid = arr[~np.isnan(arr)]
            if len(arr_valid) == 0 or np.isnan(val):
                return "N/A"
            low = np.percentile(arr_valid, 2.5)
            high = np.percentile(arr_valid, 97.5)
            return f"{val:{fmt}} [{low:{fmt}}, {high:{fmt}}]"
            
        rows.append({
            "Model": model_name,
            "Accuracy": get_ci_str("Accuracy"),
            "Macro-F1": get_ci_str("Macro-F1"),
            "Balanced Acc": get_ci_str("Balanced Acc"),
            "ROC-AUC (High)": get_ci_str("ROC-AUC (High)"),
            f"PR-AUC [base:{base_rate:.3f}]": get_ci_str("PR-AUC (High)"),
            "Spearman": get_ci_str("Spearman"),
        })
    return pd.DataFrame(rows)

print("=== Full Test Set (N=2,525) ===")
display(evaluate_models_slice(np.ones(len(test_df), dtype=bool), "Full Test Set"))

print("\n=== complete_info Subset (N=1,022) ===")
display(evaluate_models_slice(test_df["complete_info"].values == 1, "complete_info Subset"))

print("\n=== complete_info_strict Subset (N=635) ===")
display(evaluate_models_slice(test_df["complete_info_strict"].values == 1, "complete_info_strict Subset"))

=== Full Test Set (N=2,525) ===


,Model,Accuracy,Macro-F1,Balanced Acc,ROC-AUC (High),PR-AUC [base:0.253],Spearman
0,Page Completeness Baseline,"0.528 [0.509, 0.548]","0.502 [0.483, 0.524]","0.491 [0.472, 0.512]","0.736 [0.714, 0.754]","0.408 [0.375, 0.437]","0.516 [0.481, 0.548]"
1,1. core,"0.646 [0.627, 0.668]","0.635 [0.616, 0.656]","0.622 [0.603, 0.644]","0.852 [0.836, 0.867]","0.642 [0.606, 0.682]","0.715 [0.696, 0.733]"
2,2. core + talent_reach (Headline),"0.659 [0.637, 0.677]","0.656 [0.633, 0.674]","0.651 [0.627, 0.671]","0.863 [0.846, 0.878]","0.675 [0.638, 0.711]","0.752 [0.735, 0.768]"
3,3. core + talent_reach + missingness,"0.669 [0.650, 0.687]","0.668 [0.648, 0.685]","0.664 [0.643, 0.682]","0.867 [0.851, 0.881]","0.685 [0.649, 0.720]","0.770 [0.754, 0.784]"
4,4. core + talent_reach + talent_exp,"0.675 [0.658, 0.695]","0.670 [0.652, 0.689]","0.663 [0.644, 0.683]","0.878 [0.863, 0.892]","0.694 [0.658, 0.727]","0.788 [0.773, 0.801]"
5,5. core + talent_reach + text,"0.665 [0.645, 0.684]","0.662 [0.640, 0.681]","0.654 [0.633, 0.674]","0.868 [0.853, 0.884]","0.691 [0.654, 0.727]","0.770 [0.755, 0.787]"
6,6. talent_reach only,"0.535 [0.514, 0.553]","0.463 [0.442, 0.482]","0.470 [0.451, 0.486]","0.769 [0.747, 0.788]","0.516 [0.474, 0.555]","0.561 [0.533, 0.585]"
7,7. all groups,"0.678 [0.662, 0.697]","0.673 [0.654, 0.692]","0.664 [0.645, 0.684]","0.882 [0.868, 0.895]","0.709 [0.674, 0.743]","0.802 [0.788, 0.817]"
8,8. core + talent_reach (Drop 2015),"0.651 [0.631, 0.671]","0.649 [0.629, 0.669]","0.647 [0.625, 0.668]","0.867 [0.852, 0.881]","0.682 [0.646, 0.718]","0.760 [0.742, 0.776]"
9,9. core + talent_reach (V1 rolling),"0.660 [0.638, 0.679]","0.656 [0.634, 0.675]","0.651 [0.626, 0.671]","0.868 [0.852, 0.883]","0.688 [0.652, 0.724]","0.764 [0.747, 0.780]"



=== complete_info Subset (N=1,022) ===


,Model,Accuracy,Macro-F1,Balanced Acc,ROC-AUC (High),PR-AUC [base:0.389],Spearman
0,Page Completeness Baseline,"0.489 [0.459, 0.520]","0.341 [0.320, 0.363]","0.374 [0.352, 0.395]","0.569 [0.538, 0.599]","0.427 [0.392, 0.465]","0.168 [0.106, 0.230]"
1,1. core,"0.679 [0.652, 0.711]","0.659 [0.623, 0.695]","0.637 [0.600, 0.680]","0.796 [0.770, 0.828]","0.704 [0.660, 0.751]","0.632 [0.592, 0.671]"
2,2. core + talent_reach (Headline),"0.681 [0.654, 0.709]","0.675 [0.639, 0.709]","0.649 [0.612, 0.687]","0.811 [0.787, 0.835]","0.739 [0.702, 0.779]","0.664 [0.627, 0.700]"
3,3. core + talent_reach + missingness,"0.671 [0.644, 0.698]","0.662 [0.623, 0.700]","0.638 [0.600, 0.679]","0.818 [0.792, 0.843]","0.751 [0.714, 0.791]","0.678 [0.640, 0.714]"
4,4. core + talent_reach + talent_exp,"0.695 [0.666, 0.721]","0.660 [0.620, 0.699]","0.630 [0.593, 0.669]","0.834 [0.808, 0.859]","0.760 [0.723, 0.801]","0.703 [0.666, 0.736]"
5,5. core + talent_reach + text,"0.659 [0.631, 0.689]","0.625 [0.587, 0.664]","0.598 [0.562, 0.635]","0.816 [0.792, 0.841]","0.748 [0.712, 0.788]","0.672 [0.633, 0.707]"
6,6. talent_reach only,"0.593 [0.564, 0.624]","0.566 [0.526, 0.607]","0.538 [0.504, 0.578]","0.721 [0.691, 0.750]","0.595 [0.548, 0.647]","0.500 [0.448, 0.549]"
7,7. all groups,"0.692 [0.664, 0.718]","0.641 [0.597, 0.678]","0.613 [0.574, 0.650]","0.838 [0.813, 0.861]","0.767 [0.731, 0.807]","0.710 [0.672, 0.744]"
8,8. core + talent_reach (Drop 2015),"0.678 [0.651, 0.707]","0.662 [0.625, 0.697]","0.636 [0.601, 0.676]","0.813 [0.788, 0.837]","0.741 [0.705, 0.783]","0.667 [0.628, 0.703]"
9,9. core + talent_reach (V1 rolling),"0.685 [0.660, 0.713]","0.679 [0.644, 0.715]","0.655 [0.619, 0.695]","0.821 [0.796, 0.847]","0.753 [0.717, 0.791]","0.683 [0.648, 0.719]"



=== complete_info_strict Subset (N=635) ===


,Model,Accuracy,Macro-F1,Balanced Acc,ROC-AUC (High),PR-AUC [base:0.441],Spearman
0,Page Completeness Baseline,"0.441 [0.398, 0.480]","0.204 [0.190, 0.216]","0.333 [0.333, 0.333]","0.500 [0.500, 0.500]","0.441 [0.398, 0.480]",N/A
1,1. core,"0.690 [0.652, 0.723]","0.669 [0.609, 0.718]","0.637 [0.577, 0.690]","0.807 [0.772, 0.839]","0.745 [0.685, 0.798]","0.655 [0.602, 0.700]"
2,2. core + talent_reach (Headline),"0.676 [0.638, 0.711]","0.643 [0.584, 0.697]","0.613 [0.560, 0.667]","0.821 [0.788, 0.854]","0.792 [0.745, 0.836]","0.676 [0.625, 0.723]"
3,3. core + talent_reach + missingness,"0.665 [0.622, 0.699]","0.632 [0.569, 0.684]","0.605 [0.546, 0.658]","0.825 [0.792, 0.857]","0.796 [0.748, 0.841]","0.689 [0.643, 0.735]"
4,4. core + talent_reach + talent_exp,"0.693 [0.657, 0.728]","0.625 [0.563, 0.670]","0.595 [0.545, 0.639]","0.839 [0.810, 0.868]","0.803 [0.752, 0.846]","0.705 [0.659, 0.750]"
5,5. core + talent_reach + text,"0.671 [0.632, 0.709]","0.597 [0.535, 0.656]","0.569 [0.520, 0.622]","0.830 [0.800, 0.861]","0.803 [0.753, 0.845]","0.692 [0.646, 0.739]"
6,6. talent_reach only,"0.600 [0.558, 0.642]","0.533 [0.478, 0.595]","0.509 [0.463, 0.564]","0.730 [0.689, 0.770]","0.661 [0.602, 0.724]","0.509 [0.447, 0.572]"
7,7. all groups,"0.699 [0.663, 0.732]","0.602 [0.539, 0.658]","0.577 [0.529, 0.624]","0.845 [0.815, 0.873]","0.809 [0.760, 0.850]","0.716 [0.674, 0.760]"
8,8. core + talent_reach (Drop 2015),"0.672 [0.631, 0.709]","0.627 [0.566, 0.680]","0.599 [0.546, 0.652]","0.820 [0.786, 0.851]","0.789 [0.743, 0.832]","0.672 [0.623, 0.719]"
9,9. core + talent_reach (V1 rolling),"0.677 [0.641, 0.711]","0.641 [0.580, 0.691]","0.614 [0.559, 0.664]","0.830 [0.799, 0.859]","0.803 [0.759, 0.844]","0.695 [0.648, 0.741]"


## Step 3: Probability Calibration (Fitted on Internal Validation Only)

In [6]:
# Calibration: Fit on internal validation (<=2019 train vs 2020-2021 val)
headline_cols = feature_sets["2. core + talent_reach (Headline)"]
y_val_bin = (val_sub["target_tier"].values == "High").astype(int)

# Predict on val
_, val_probs = train_and_predict_hgb(tr_sub, val_sub, headline_cols, params=HGB_TUNED_PARAMS)

# Fit Platt and Isotonic calibrators
platt = LogisticRegression(random_state=42).fit(val_probs.reshape(-1, 1), y_val_bin)
iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(val_probs, y_val_bin)

# Apply to test
_, test_probs_raw = predictions["2. core + talent_reach (Headline)"]
test_probs_platt = platt.predict_proba(test_probs_raw.reshape(-1, 1))[:, 1]
test_probs_iso = iso.predict(test_probs_raw)

y_test_bin = (test_df["target_tier"].values == "High").astype(int)
base_rate = float(y_test_bin.mean())
brier_baseline = base_rate * (1 - base_rate)

def eval_cal(probs, name):
    brier = float(np.mean((probs - y_test_bin)**2))
    bins = np.linspace(0.0, 1.0, 11)
    bin_ids = np.digitize(probs, bins) - 1
    ece = 0.0
    rows = []
    for b in range(10):
        mask = bin_ids == b
        nb = int(mask.sum())
        if nb == 0:
            continue
        mp = float(probs[mask].mean())
        mo = float(y_test_bin[mask].mean())
        ece += (nb / len(probs)) * abs(mp - mo)
        rows.append({"Bin": f"[{bins[b]:.1f}, {bins[b+1]:.1f})", "N": nb, "Pred": f"{mp:.3f}", "Obs": f"{mo:.3f}", "Diff": f"{mp-mo:+.3f}"})
    return brier, ece, pd.DataFrame(rows)

brier_raw, ece_raw, t_raw = eval_cal(test_probs_raw, "Raw")
brier_platt, ece_platt, t_platt = eval_cal(test_probs_platt, "Platt")
brier_iso, ece_iso, t_iso = eval_cal(test_probs_iso, "Isotonic")

print(f"Brier baseline (predicting base rate {base_rate:.3f}): {brier_baseline:.4f}")
print(f"Raw Model:   Brier = {brier_raw:.4f}, ECE = {ece_raw:.4f}")
print(f"Platt Scaled: Brier = {brier_platt:.4f}, ECE = {ece_platt:.4f}")
print(f"Isotonic:     Brier = {brier_iso:.4f}, ECE = {ece_iso:.4f}")
display(t_iso)

Brier baseline (predicting base rate 0.253): 0.1888
Raw Model:   Brier = 0.1309, ECE = 0.0493
Platt Scaled: Brier = 0.1292, ECE = 0.0443
Isotonic:     Brier = 0.1272, ECE = 0.0318


,Bin,N,Pred,Obs,Diff
0,"[0.0, 0.1)",1059,0.022,0.027,-0.006
1,"[0.1, 0.2)",373,0.139,0.190,-0.052
2,"[0.2, 0.3)",531,0.246,0.315,-0.068
3,"[0.3, 0.4)",2,0.322,0.500,-0.178
4,"[0.5, 0.6)",242,0.545,0.570,-0.025
5,"[0.6, 0.7)",107,0.604,0.598,+0.006
6,"[0.7, 0.8)",52,0.723,0.712,+0.011
7,"[0.8, 0.9)",94,0.845,0.766,+0.079
8,"[0.9, 1.0)",53,0.953,0.887,+0.066
